transformers → Provides pre-trained models and APIs to load, fine-tune, and run LLMs.

datasets → Easy access and processing of large datasets for training and evaluation.

peft → Enables parameter-efficient fine-tuning methods like LoRA without updating full model weights.

accelerate → Simplifies multi-GPU, mixed precision, and distributed training.

bitsandbytes → Enables low-bit quantization (4-bit/8-bit) to reduce memory usage.

trl → Provides tools like SFTTrainer for training LLMs using reinforcement learning and supervised fine-tuning.

In [ ]:
!pip install -U transformers datasets peft accelerate bitsandbytes trl

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

In [ ]:
# MODEL_NAME = "google/gemma-2-9b-it"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./TinyLlama"

In [ ]:
dataset = load_dataset("Abirate/english_quotes")

In [ ]:
dataset

In [ ]:
def format_example(example):
    quote = example["quote"].strip()
    tags = ", ".join(example["tags"]) if isinstance(example["tags"], list) else str(example["tags"])

    text = f"""<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
{quote}<end_of_turn>
<start_of_turn>model
{tags}<end_of_turn>"""
    return {"text": text}


In [ ]:
train_dataset = dataset["train"].map(format_example)

In [ ]:
train_dataset

#### This format teaches the model how to respond to a user instruction by mapping input text to the expected output.

In [ ]:
print(train_dataset['text'][0])

In [ ]:
print(train_dataset['text'][1])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
total_params = model.num_parameters()
trainable_params = model.num_parameters(only_trainable=True)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable %:", 100 * trainable_params / total_params)

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape, param.dtype)

In [ ]:
print(model.peft_config)

In [ ]:
config = model.peft_config["default"]

print("LoRA rank:", config.r)
print("LoRA alpha:", config.lora_alpha)
print("LoRA dropout:", config.lora_dropout)
print("Target modules:", config.target_modules)
print("Task type:", config.task_type)

In [ ]:
print(base_model.config.quantization_config)

In [ ]:
print("Loaded in 4-bit:", base_model.is_loaded_in_4bit)
print("Device map:", getattr(base_model, "hf_device_map", None))

In [ ]:
print("Memory footprint in GB:", model.get_memory_footprint() / 1024**3)

In [ ]:
for name, param in model.named_parameters():
    print(name, param.dtype, param.device, param.requires_grad)
    break

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

In [ ]:
trainer.train()

In [ ]:
#saving lora adapter
model.save_pretrained(f"{OUTPUT_DIR}/adapter")

In [ ]:
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")

In [ ]:
base_model=MODEL_NAME

In [ ]:
base_model

In [ ]:
inference_model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/adapter")

In [ ]:
inference_model.eval()

In [ ]:
prompt = """<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
If you want to achieve greatness, stop asking for permission.<end_of_turn>
<start_of_turn>model
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)

In [ ]:
with torch.no_grad():
    output = inference_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

In [ ]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
#saving full model
merged_model = inference_model.merge_and_unload()

In [ ]:
tokenizer.save_pretrained("./merged_full_model")

In [ ]:
merged_model.save_pretrained("./merged_full_model")

In [ ]:
for i in range(n):
    c[i]=a[i]+b[i]